<img class="nvidia-header-light" src="images/nvidia_header_black.png" style="margin-left: -30px; width: 300px; float: left;">
<img class="nvidia-header-dark" src="images/nvidia_header_white.png" style="margin-left: -30px; width: 300px; float: left;">

# Part 4 — Action Generation with Cosmos 3

Where the reasoner *understands* a scene (Part 2) and the world model *generates* it (Part 3), the **action** capability lets Cosmos 3 *act* in it — turning perception and reasoning into embodied decisions and control. This completes the **perceive → reason → act** loop at the heart of Physical AI.

Cosmos 3 treats **action as a modality**: action tokens represent the transition between consecutive visual states. The same omni-model is used in three action modes:

| Mode | Given | Predicts | Question it answers |
| --- | --- | --- | --- |
| **Forward dynamics** | a start image + an action trajectory | future video | *"If I take these actions, what will I see?"* |
| **Inverse dynamics** | a video | the action trajectory | *"What actions produced this motion?"* |
| **Policy** | a start observation + a task instruction | future video **and** an action sequence | *"What should I do to achieve this goal?"* |

This notebook runs in the **"Cosmos 3 (framework)"** kernel. All three modes run hands-on below as one-shot `cosmos_framework.scripts.inference` jobs on the same `Cosmos3-Nano` checkpoint. Closed-loop policy control — a streaming policy server plus the RoboLab simulator — is described as an advanced, out-of-lab step at the end.

## 1. Environment

As in Part 3, the Cosmos Framework is pre-installed at `/opt/cosmos3-framework` and this kernel is its virtual environment. The cell below configures paths, the GPU, and a `resolve_input` helper for the cookbook's action assets. Action inference uses the latency parallelism preset on a single GPU.

In [ ]:
import os
import sys
import json
import socket
from pathlib import Path

def free_local_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return str(s.getsockname()[1])

COSMOS_ROOT = Path(os.environ.get("COSMOS_ROOT", "/dli/task/cosmos")).resolve()
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", "/opt/cosmos3-framework")).resolve()
ACTION_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "action"
OUTPUT_ROOT = Path(os.environ.get("COSMOS3_OUTPUT_ROOT", COSMOS3_REPO / "outputs" / "action"))
INPUT_DIR = OUTPUT_ROOT / "inputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_input(rel_path):
    p = (COSMOS_ROOT / rel_path).resolve()
    assert p.exists(), f"missing input: {p}"
    return str(p)

# Make `import cosmos_framework` work for the visualization helpers below.
if str(COSMOS3_REPO) not in sys.path:
    sys.path.insert(0, str(COSMOS3_REPO))

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")
os.environ.setdefault("COSMOS3_MASTER_ADDR", "127.0.0.1")
os.environ.setdefault("COSMOS3_MASTER_PORT", free_local_port())
os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))
os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_CHECKPOINT_PATH"] = os.environ.get("COSMOS3_CHECKPOINT_PATH", "Cosmos3-Nano")

for p in [COSMOS3_REPO, ACTION_ROOT]:
    assert p.exists(), f"missing: {p}"

print("Cosmos Framework:", COSMOS3_REPO)
print("Action assets:   ", ACTION_ROOT / "assets")
print("Output root:     ", OUTPUT_ROOT)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("checkpoint:", os.environ["COSMOS3_CHECKPOINT_PATH"])

In [ ]:
import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  device {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# Compact MP4 preview helper (full-resolution outputs can freeze the front-end).
import base64
import subprocess
import urllib.request
import imageio_ffmpeg
from IPython.display import HTML, Image, display

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

# Model outputs live under OUTPUT_ROOT (/opt/cosmos3-framework/outputs), which is
# OUTSIDE the notebook root, so the front-end can't fetch them by URL -- and in
# VS Code there is no Jupyter file server at all (the kernel is launched directly),
# so a `/lab/files/...` or remote https <video src> renders as an empty box.
# Instead we transcode a small, low-bitrate copy (a few hundred KB at most) and
# embed it inline as a base64 data URI, which plays in JupyterLab and VS Code alike.
_NB_ROOT = Path("/dli/task")                 # notebook root
_PREVIEW_DIR = _NB_ROOT / "assets" / "_previews"

def preview(src, crf=28, width=480):
    """Show a local path or http(s) URL of an MP4 as an inline, compact video."""
    _PREVIEW_DIR.mkdir(parents=True, exist_ok=True)
    src = str(src)
    if "://" in src:                          # remote clip: fetch it once
        local = _PREVIEW_DIR / ("remote__" + Path(src.split("?")[0]).name)
        if not local.exists():
            urllib.request.urlretrieve(src, local)
        label, src = src, local
    else:
        src = Path(src).resolve()
        label = str(src)
    # Every run writes `vision.mp4`, so key the cache on the parent dir too
    # (av_forward/vision.mp4 vs robot_policy/vision.mp4 must not collide).
    out = _PREVIEW_DIR / f"{src.parent.name}__{src.stem}_preview.mp4"
    if not out.exists() or out.stat().st_mtime < src.stat().st_mtime:
        # -movflags +faststart puts the moov atom up front so playback can start
        # before the whole file is buffered.
        subprocess.run([FFMPEG, "-y", "-loglevel", "error", "-i", str(src),
                        "-c:v", "libx264", "-crf", str(crf), "-preset", "veryfast",
                        "-vf", f"scale={width}:-2",
                        "-an", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
                        str(out)], check=True)
    b64 = base64.b64encode(out.read_bytes()).decode("ascii")
    size_mb = src.stat().st_size / 1e6
    display(HTML(
        f'<video controls playsinline loop width="{width}" style="background:#000;display:block" '
        f'src="data:video/mp4;base64,{b64}"></video>'
        f'<div style="font:12px monospace;opacity:.75;margin:4px 0 10px">{label} ({size_mb:.1f} MB)</div>'))

print("preview helper ready")


## 2. Forward dynamics — predict video from an image and an action trajectory

Forward dynamics answers *"if I take these actions, what will I see?"*. We give Cosmos 3 a single start frame from an autonomous-vehicle camera plus an ego-motion trajectory (a sequence of 9D pose deltas), and it generates the driving video that those actions would produce.

The cookbook ships three AV trajectories (`forward`, `left`, `right`) that all start from the same frame. We run the **forward** trajectory below; try `av_traj_left.json` or `av_traj_right.json` to see the same scene steered differently.

In [ ]:
# Start frame for the AV forward-dynamics run.
fd_image = "cookbooks/cosmos3/generator/action/assets/images/av_0.jpg"
fd_action = "cookbooks/cosmos3/generator/action/assets/actions/av_traj_forward.json"

# Show the start frame the model will predict future video from.
display(Image(filename=resolve_input(fd_image), width=480))

fd_record = {
    "action_chunk_size": 60,
    "action_path": resolve_input(fd_action),
    "domain_name": "av",
    "fps": 10,
    "image_size": 480,
    "view_point": "ego_view",
    "model_mode": "forward_dynamics",
    "name": "av_forward",
    "prompt": "You are an autonomous vehicle planning system.",
    "seed": 0,
    "vision_path": resolve_input(fd_image),
}

fd_input = INPUT_DIR / "action_forward_dynamics_av.jsonl"
fd_input.write_text(json.dumps(fd_record) + "\n")
fd_output = OUTPUT_ROOT / "action_forward_dynamics_av"
os.environ["FD_INPUT"] = str(fd_input)
os.environ["FD_OUTPUT"] = str(fd_output)
print("wrote spec:", fd_input)
print(fd_input.read_text())

Run forward dynamics. The first run downloads the `Cosmos3-Nano` weights from Hugging Face (gated — complete notebook 01 first). The generated video is written to `<output>/av_forward/vision.mp4`.

In [ ]:
%%bash
set -euo pipefail
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
MASTER_ADDR="$COSMOS3_MASTER_ADDR" MASTER_PORT="$COSMOS3_MASTER_PORT" RANK=0 WORLD_SIZE=1 LOCAL_RANK=0 \
.venv/bin/python -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  -i "$FD_INPUT" \
  -o "$FD_OUTPUT" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --video-save-quality 8 \
  --image_size 480 \
  --seed 0 \
  2>&1 | grep --line-buffered -vE "OmniMoTModel: config|hf download|Installed [0-9]+ packages"
# ^ the framework writes the complete log to <output>/console.log; the filter only
#   hides the multi-KB config dump and download chatter from the notebook output.

In [ ]:
preview(fd_output / "av_forward" / "vision.mp4")

## 3. Inverse dynamics — predict the action trajectory from a video

Inverse dynamics is the reverse: given a video, predict the ego-motion trajectory that produced it. Instead of a video, the model outputs an **action** — a sequence of 9D pose deltas — which we then convert back into camera poses and visualize as a 3D path and a bird's-eye view.

We provide an AV clip and **no** `action_path` (the action is what the model predicts).

In [ ]:
id_video = "cookbooks/cosmos3/generator/action/assets/videos/av_0.mp4"
id_record = {
    "action_chunk_size": 60,
    "domain_name": "av",
    "fps": 10,
    "image_size": 480,
    "view_point": "ego_view",
    "model_mode": "inverse_dynamics",
    "name": "av_inverse_0",
    "prompt": "You are an autonomous vehicle planning system.",
    "seed": 0,
    "vision_path": resolve_input(id_video),
}

id_input = INPUT_DIR / "action_inverse_dynamics_av.jsonl"
id_input.write_text(json.dumps(id_record) + "\n")
id_output = OUTPUT_ROOT / "action_inverse_dynamics_av"
os.environ["ID_INPUT"] = str(id_input)
os.environ["ID_OUTPUT"] = str(id_output)
print("wrote spec:", id_input)
print("input video:")
preview(resolve_input(id_video))

In [ ]:
%%bash
set -euo pipefail
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
MASTER_ADDR="$COSMOS3_MASTER_ADDR" MASTER_PORT="$COSMOS3_MASTER_PORT" RANK=0 WORLD_SIZE=1 LOCAL_RANK=0 \
.venv/bin/python -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  -i "$ID_INPUT" \
  -o "$ID_OUTPUT" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed 0 \
  2>&1 | grep --line-buffered -vE "OmniMoTModel: config|hf download|Installed [0-9]+ packages"
# ^ the framework writes the complete log to <output>/console.log; the filter only
#   hides the multi-KB config dump and download chatter from the notebook output.

The predicted action is stored under `outputs[0].content["action"]` as `[T-1, 9]` relative pose deltas. We convert it back to absolute camera poses with the framework's `pose_rel_to_abs` and plot the trajectory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from cosmos_framework.data.generator.action.utils.pose_utils import pose_rel_to_abs

# frustum: apex + image-rectangle corners (camera +Z forward), and their edges
_FRUSTUM = np.array([[0, 0, 0], [-1, -1, 1], [1, -1, 1], [1, 1, 1], [-1, 1, 1]], float)
_EDGES = [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (2, 3), (3, 4), (4, 1)]

def visualize_pose(poses_abs, *, n_frustums=20, scale_frac=0.03, aspect=16 / 9,
                   fov_deg=60.0, cmap="turbo", title=None):
    """3D camera trajectory (with frustums) + a top-down bird's-eye view."""
    poses_abs = np.asarray(poses_abs)
    pos = poses_abs[:, :3, 3]      # camera centers
    fwd = poses_abs[:, :3, 2]      # heading (+Z)
    T = len(pos)
    colors = plt.get_cmap(cmap)(np.arange(T) / max(T - 1, 1))
    scale = max(np.ptp(pos, axis=0).max() * scale_frac, 1e-3)
    step = max(1, T // max(n_frustums, 1))
    xzy = [0, 2, 1]               # world (X,Y,Z) -> plot (X, Z, Y-up)

    fig = plt.figure(figsize=(14, 6))
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    path = pos[:, xzy]
    ax.plot(*path.T, color="0.6", lw=1.0, alpha=0.7)
    lines, lcolors, allpts = [], [], [path]
    for i in range(0, T, step):
        cw = ((_FRUSTUM * [aspect, 1, 1] * scale * np.tan(np.radians(fov_deg) / 2))
              @ poses_abs[i, :3, :3].T + poses_abs[i, :3, 3])[:, xzy]
        allpts.append(cw)
        lines += [[cw[a], cw[b]] for a, b in _EDGES]
        lcolors += [colors[i]] * len(_EDGES)
    ax.add_collection3d(Line3DCollection(lines, colors=lcolors, linewidths=1.2))
    ax.scatter(*path[0], color="lime", s=80, edgecolor="k", label="first frame", zorder=5)
    ax.scatter(*path[-1], color="red", s=80, edgecolor="k", label="last frame", zorder=5)
    rng = np.clip(np.ptp(np.concatenate(allpts), axis=0), 1e-9, None)
    ax.set_box_aspect((rng[0], rng[1], rng[2]))
    ax.set_xlabel("X (m)"); ax.set_ylabel("Z forward (m)"); ax.set_zlabel("Y up (m)")
    ax.set_zticks([]); ax.set_title(title or f"Camera trajectory ({T} frames)")
    ax.legend(loc="upper left"); ax.view_init(elev=22, azim=-70)

    ax2 = fig.add_subplot(1, 2, 2)
    seg = np.stack([pos[:-1, [0, 2]], pos[1:, [0, 2]]], axis=1)
    lc = LineCollection(seg, cmap=cmap, norm=plt.Normalize(0, T - 1), linewidth=2.5)
    lc.set_array(np.arange(T - 1)); ax2.add_collection(lc)
    ax2.quiver(pos[::step, 0], pos[::step, 2], fwd[::step, 0], fwd[::step, 2],
               color=colors[::step], angles="xy", width=0.005, scale=22, zorder=3)
    ax2.scatter(*pos[0, [0, 2]], color="lime", s=80, edgecolor="k", zorder=5)
    ax2.scatter(*pos[-1, [0, 2]], color="red", s=80, edgecolor="k", zorder=5)
    ax2.set_xlabel("X (m)"); ax2.set_ylabel("Z forward (m)")
    ax2.set_title("Top-down (bird's-eye view)")
    ax2.set_aspect("equal", adjustable="datalim"); ax2.autoscale_view()
    fig.colorbar(lc, ax=ax2, label="frame index")
    plt.tight_layout(w_pad=6); plt.show()

outputs = json.loads((id_output / "av_inverse_0" / "sample_outputs.json").read_text())
poses_rel = np.array(outputs["outputs"][0]["content"]["action"])  # [T-1, 9]
poses_abs = pose_rel_to_abs(poses_rel, rotation_format="rot6d",
                            pose_convention="backward_framewise", translation_scale=1.35)
print("predicted action:", poses_rel.shape, "-> camera poses:", poses_abs.shape)
visualize_pose(poses_abs, title=f"Predicted camera trajectory ({len(poses_abs)} frames)")

## 4. Policy mode

The third action mode is **policy**: given a start observation and a natural-language task instruction, Cosmos 3 predicts both the future video **and** the action sequence to achieve the goal. This is what turns scene understanding into executable robot behavior — the agent carries out a *described* task rather than a fixed, pre-scripted motion, and the actions stay grounded in physics and the scene.

Like forward and inverse dynamics, policy runs as a **one-shot inference** with `model_mode: wam` (the framework's name for the policy / *world-action model* mode) on the same `Cosmos3-Nano` checkpoint — no extra service. The clip below is an example of a *closed-loop* policy rollout in the [RoboLab](https://github.com/NVlabs/RoboLab) simulator (the advanced setup covered at the end); first we run the open-loop, single-shot policy directly.

In [ ]:
rollout = ACTION_ROOT / "assets" / "videos" / "robolab_example_rollout.mp4"
if rollout.exists():
    preview(rollout, width=560)
else:
    print("Example rollout not found:", rollout)

### Run the policy model

We drive `cosmos_framework.scripts.inference` with `model_mode: wam` on `Cosmos3-Nano` — the same entrypoint used for forward/inverse dynamics. We use the framework's canonical **robot manipulation** example: from a start observation of a tabletop scene and the instruction *"Put the pot to the left of the purple item."*, the model predicts the **end-effector action sequence** (`sample_outputs.json`) and a **future-rollout video** (`vision.mp4`).

The start observation is fetched from the Cosmos dependencies repo, so this cell needs network access.

In [ ]:
# Build the policy input spec (robot manipulation, model_mode=wam). No
# action_path: the action is what the model predicts. The start observation is
# the framework's canonical Bridge example, fetched over the network.
policy_video = ("https://github.com/nvidia-cosmos/cosmos-dependencies/raw/"
                "2b17a2413bd86b2cf9b03823637108851e4ddf2d/inputs/action/bridge_20260501_0.mp4")

# Show the start observation (downloaded once and previewed inline)
preview(policy_video)

policy_record = {
    "action_chunk_size": 16,
    "domain_name": "bridge_orig_lerobot",
    "fps": 5,
    "image_size": 480,
    "view_point": "ego_view",
    "model_mode": "wam",   # the framework's name for policy mode (world-action model)
    "name": "robot_policy",
    "prompt": "Put the pot to the left of the purple item.",
    "seed": 0,
    "vision_path": policy_video,
}

policy_input = INPUT_DIR / "action_policy_robot.jsonl"
policy_input.write_text(json.dumps(policy_record) + "\n")
policy_output = OUTPUT_ROOT / "action_policy_robot"
os.environ["POLICY_INPUT"] = str(policy_input)
os.environ["POLICY_OUTPUT"] = str(policy_output)
print("wrote spec:", policy_input)
print(policy_input.read_text())

In [ ]:
%%bash
set -euo pipefail
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
MASTER_ADDR="$COSMOS3_MASTER_ADDR" MASTER_PORT="$COSMOS3_MASTER_PORT" RANK=0 WORLD_SIZE=1 LOCAL_RANK=0 \
.venv/bin/python -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  -i "$POLICY_INPUT" \
  -o "$POLICY_OUTPUT" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed 0 \
  2>&1 | grep --line-buffered -vE "OmniMoTModel: config|hf download|Installed [0-9]+ packages"
# ^ the framework writes the complete log to <output>/console.log; the filter only
#   hides the multi-KB config dump and download chatter from the notebook output.

### Inspect the predicted rollout and action

Policy predicts **both** outputs: a future-rollout video (`vision.mp4`) and the end-effector action sequence (`sample_outputs.json`). Let's view both.

In [ ]:
import json
import numpy as np

policy_dir = policy_output / "robot_policy"

rollout_mp4 = policy_dir / "vision.mp4"
if rollout_mp4.exists():
    print("Predicted rollout video:")
    preview(rollout_mp4)
else:
    print("No rollout video found at", rollout_mp4)

sample_json = policy_dir / "sample_outputs.json"
if sample_json.exists():
    out = json.loads(sample_json.read_text())
    action = np.asarray(out["outputs"][0]["content"]["action"])
    print("predicted action sequence shape:", action.shape)
    print("first 3 steps:\n", np.round(action[:3], 4))
else:
    print("No sample_outputs.json found at", sample_json)

### Draw the predicted trajectory in 3D

The robot policy predicts an **end-effector** action — per step: a 3D translation delta, a 6D rotation, and a gripper open/close value. Integrating the translation deltas traces the gripper's **path through 3D space**, which we plot (colored by time) next to the predicted gripper state over the sequence.

In [ ]:
# Draw the policy's predicted end-effector path in 3D (self-contained).
import json
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection

policy_sample = policy_output / "robot_policy" / "sample_outputs.json"
if not policy_sample.exists():
    print("No policy action found - run the policy inference cell above first.")
else:
    action = np.asarray(json.loads(policy_sample.read_text())["outputs"][0]["content"]["action"])  # [T, D]
    # Action layout: dims 0:3 = end-effector translation deltas (m), 3:9 = 6D
    # rotation, 9 = gripper. Integrate the translation deltas into a 3D path.
    positions = np.cumsum(action[:, :3], axis=0)
    gripper = action[:, 9] if action.shape[1] > 9 else None
    T = len(positions)
    colors = plt.get_cmap("turbo")(np.linspace(0, 1, T))
    print("predicted action:", action.shape, "-> end-effector path:", positions.shape)

    fig = plt.figure(figsize=(13, 5))
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    if T > 1:
        seg = np.stack([positions[:-1], positions[1:]], axis=1)
        ax.add_collection3d(Line3DCollection(seg, colors=colors[:-1], linewidths=2.5))
    ax.scatter(*positions[0], color="lime", s=90, edgecolor="k", label="start", zorder=5)
    ax.scatter(*positions[-1], color="red", s=90, edgecolor="k", label="end", zorder=5)
    ax.set_box_aspect(tuple(np.clip(np.ptp(positions, axis=0), 1e-6, None)))
    ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.set_zlabel("Z (m)")
    ax.set_title(f"Predicted end-effector path ({T} steps)")
    ax.legend(loc="upper left"); ax.view_init(elev=22, azim=-60)

    ax2 = fig.add_subplot(1, 2, 2)
    if gripper is not None:
        ax2.plot(gripper, color="tab:blue")
        ax2.set_xlabel("step"); ax2.set_ylabel("gripper (open → close)")
        ax2.set_title("Predicted gripper state")
    else:
        ax2.axis("off")
    plt.tight_layout(w_pad=4)
    plt.show()

### Closed-loop control with RoboLab (advanced, out-of-lab)

The run above is **open-loop**: one observation in, one predicted action sequence (and rollout) out. **Closed-loop** control instead steps a simulator and re-queries the policy every chunk, feeding each new observation back in. NVIDIA exposes this through a streaming **policy server** for the post-trained [`Cosmos3-Nano-Policy-DROID`](https://huggingface.co/nvidia/Cosmos3-Nano-Policy-DROID) checkpoint, with [RoboLab](https://github.com/NVlabs/RoboLab) (Isaac Sim 5.0 + Isaac Lab 2.2.0, runs headless) as the client.

That setup needs the framework's `policy-server` extras and a full Isaac Sim install, so it is an **out-of-lab exercise** rather than part of this notebook. See the framework's [policy server guide](https://github.com/NVIDIA/cosmos-framework/blob/main/docs/action_policy_droid_server.md): start `cosmos_framework.scripts.action_policy_server_robolab`, then run a RoboLab task (e.g. `policies/cosmos3/run.py --task BananaInBowlTask`) against it.

Conceptually this closes the **perceive → reason → act** loop: the reasoner (Part 2) supplies scene understanding and high-level plans, the world model (Part 3) supplies large-scale augmented training data, and the action model consumes both to learn and execute a robust policy.

## Wrapping up

Across these notebooks you have used a single Cosmos 3 omni-model to **reason** about video, **generate** and augment it, and **act** through forward dynamics, inverse dynamics, and policy. Together they form the end-to-end Physical AI data and control loop: perceive a scene, reason about and augment it to scale training data efficiently, and translate that understanding into real, executable robot control.

## Resources

- [Cosmos 3 cookbook — Generator (action)](https://github.com/NVIDIA/cosmos/tree/main/cookbooks/cosmos3/generator/action)
- [Cosmos3 Action Viewer](https://huggingface.co/spaces/nvidia/Cosmos3-Action-Viewer)
- [RoboLab](https://github.com/NVlabs/RoboLab)
- [Cosmos Framework](https://github.com/NVIDIA/cosmos-framework)

<br clear="all">
<hr>
<img class="nvidia-header-light" src="images/nvidia_header_black.png" style="margin-left: -30px; width: 300px; float: left;">
<img class="nvidia-header-dark" src="images/nvidia_header_white.png" style="margin-left: -30px; width: 300px; float: left;">